# 1. View a copy v NumPy

Nejdřív importujeme `numpy` pod obvyklou zkratkou `np`.


In [1]:
import numpy as np


## 1.1 Řezy obvykle vrací view

Při běžném řezání (`:`) NumPy většinou vytvoří `view` na původní data, ne jejich kopii. Když upravíme `view`, změní se i původní pole.


In [2]:
array = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
view = array[:1, 1:]
print(array)
print(view)
view[:, :] = -1
print(array)
print(view)

[[1 2 3]
 [4 5 6]
 [7 8 9]]
[[2 3]]
[[ 1 -1 -1]
 [ 4  5  6]
 [ 7  8  9]]
[[-1 -1]]


Tohle chování je často užitečné a šetří paměť. Pokud chceme nezávislá data, použijeme `copy()`.

Ne každá indexace vrací `view`:
- běžné řezání (`:`) typicky vrací `view`,
- maskování a fancy indexing (pole indexů) typicky vrací kopii.

Atribut `flags['OWNDATA']` ukazuje, jestli pole vlastní svá data. Přes atribut `base` lze zjistit, ze kterého pole data pochází.

In [3]:
array = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
view = array[:1, 1:]
print(array.flags)
print(view.flags)

  C_CONTIGUOUS : True
  F_CONTIGUOUS : False
  OWNDATA : True
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

  C_CONTIGUOUS : True
  F_CONTIGUOUS : True
  OWNDATA : False
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False



In [4]:
print(id(array))
print(id(view.base))
print(array is view.base)

2522881357072
2522881357072
True


Pokud si nejsme jistí, je nejlepší ověřit chování v dokumentaci NumPy:
https://numpy.org/doc/stable/user/basics.indexing.html

V praxi lze sdílení paměti zkontrolovat přes `base`, `flags` nebo `np.shares_memory`.

In [5]:
array = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
mask = np.array([True, False, True])

slice_view = array[:, 1]
print("Slicing shares memory:", np.shares_memory(slice_view, array))

masked = array[mask, :]
print("Masking shares memory:", np.shares_memory(masked, array))

fancy = array[:, [1]]
print("Fancy indexing shares memory:", np.shares_memory(fancy, array))

diag = np.diag(array, k=-1)
print("Diag shares memory:", np.shares_memory(diag, array))

upper = np.triu(array)
print("Triu shares memory:", np.shares_memory(upper, array))

Slicing shares memory: True
Masking shares memory: False
Fancy indexing shares memory: False
Diag shares memory: True
Triu shares memory: False


## 1.2 Změna tvaru polí a kopírování dat

Tvar pole lze často změnit bez kopírování dat. Výsledkem je nové `ndarray`, které může sdílet stejnou paměť, ale má jiný tvar.


In [6]:
m1 = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(m1)

[[1 2 3]
 [4 5 6]
 [7 8 9]]


In [7]:
n, m = m1.shape
print(m,n)

3 3


Pokud chceme 1D vektor, můžeme použít:
- `reshape` (obvykle `view`, pokud je to možné),
- `ravel` (pokusí se vrátit `view`),
- `flatten` (vždy vrací kopii).


In [8]:
flat = m1.reshape(n * m)
print(flat)
print(flat.shape)
print(flat.base is m1)

flat = m1.reshape(-1)
print(flat)
print(flat.shape)
print(flat.base is m1)

flat = m1.flatten()
print(flat)
print(flat.shape)
print(flat.base is m1)

flat = m1.ravel()
print(flat)
print(flat.shape)
print(flat.base is m1)

[1 2 3 4 5 6 7 8 9]
(9,)
True
[1 2 3 4 5 6 7 8 9]
(9,)
True
[1 2 3 4 5 6 7 8 9]
(9,)
False
[1 2 3 4 5 6 7 8 9]
(9,)
True


In [9]:
v = np.array([1, 2, 3])
column_vector = v[:, None]
print(column_vector)
print(column_vector.base is v)

[[1]
 [2]
 [3]]
True
